# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "GISAID_Andersen_Combined_Files/"
temp_files = downloads + "Andersen_Temp_Files/"

update_date = "05-12-2025"

os.chdir(originals)

## Collection Dates

In [2]:
# Upload saved data 
os.chdir(temp_files + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific
0,0,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,...,D1.3,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas,20-Mar-2024
1,1,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,...,B3.13,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas,20-Mar-2024
2,2,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,...,B3.13,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas,20-Mar-2024
3,3,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,...,B3.13,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas,20-Mar-2024
4,4,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,...,B3.13,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4,A/cattle/Texas/24-009088-001/2024,Texas,13-Mar-2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4058,4058,SRR32633088,WGS,145.11,67016467,PRJNA1102327,SAMN47290851,Viral,25388495,USDA-NVSL,...,B3.13,SRR32633088_HA_cns.fa,Consensus_SRR32633088_HA_cns_threshold_0.5_qua...,SRR32633088,HA,PV456280.1,4,A/cat/OR/25-005913-003-original/2025,OR,2025-02-14
4059,4059,SRR32633089,WGS,148.14,140023511,PRJNA1102327,SAMN47290850,Viral,52157519,USDA-NVSL,...,B3.13,SRR32633089_HA_cns.fa,Consensus_SRR32633089_HA_cns_threshold_0.5_qua...,SRR32633089,HA,PV456272.1,4,A/cat/OR/25-005800-002-original/2025,OR,2025-02-12
4060,4060,SRR32633090,WGS,148.49,83727737,PRJNA1102327,SAMN47290849,Viral,31619233,USDA-NVSL,...,D1.3,SRR32633090_HA_cns.fa,Consensus_SRR32633090_HA_cns_threshold_0.5_qua...,SRR32633090,HA,PV456264.1,4,A/cat/OR/25-005800-001-original/2025,OR,2025-02-12
4061,4061,SRR32633093,WGS,148.10,105910536,PRJNA1102327,SAMN47290846,Viral,39757708,USDA-NVSL,...,B3.2,SRR32633093_HA_cns.fa,Consensus_SRR32633093_HA_cns_threshold_0.5_qua...,SRR32633093,HA,PV457240.1,4,A/cattle/CA/25-005677-001-original/2025,CA,2025-02-11


In [3]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2]
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header
                if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                    # Find the correct collection date, if it exists
                    row = metadata_genbank[metadata_genbank["isolate"] == isolate]
                    
                    try: # Isolate may not be in this dataset
                        biosample = row["BioSample"].values[0]
                        collection_date = search_collection_date(biosample, row) # Update unknown dates, if possible
                    except:
                        print("No biosample nor date found for isolate", isolate)
                        # no_updates_isolate.append(isolate)

                if state == "USA": # If we don't have a state
                    row = metadata_genbank[metadata_genbank["isolate"] == isolate]

                    try: # Isolate may not be in this dataset
                        genbank_name = row["genbank_name"].values[0]
                        state = genbank_name.split("/")[2]
                    except:
                        print("No state found for isolate", isolate)
                        # no_updates_isolate.append(isolate)

                updates[header] = [collection_date, state, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["date", "state", "sequence"])
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            
            header = header.replace(header.split("|")[-3], row["date"]).replace(header.split("/")[2], row["state"], 1) # Only the first instance is replaced

            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [4]:
# Create files with updates

os.chdir(originals)

for dirpath, dirs, files in os.walk(originals + "11-2023--04-14-2025_B3_13/"): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file)
        update(file_name, update_date)
    break 



2025
2025
No biosample nor date found for isolate 24-017992-001
No biosample nor date found for isolate 24-017992-002
No biosample nor date found for isolate 006543-001
No state found for isolate 006543-001
No biosample nor date found for isolate 006544-001
No state found for isolate 006544-001
No biosample nor date found for isolate 006638-001
No state found for isolate 006638-001
No biosample nor date found for isolate 007097-001
No state found for isolate 007097-001
No biosample nor date found for isolate 007097-002
No state found for isolate 007097-002
No biosample nor date found for isolate 008304-001
No state found for isolate 008304-001
No biosample nor date found for isolate 008304-003-v
No state found for isolate 008304-003-v
No biosample nor date found for isolate 008557-001
No state found for isolate 008557-001
No biosample nor date found for isolate 008596-002
No state found for isolate 008596-002
No biosample nor date found for isolate 24-035670-001
No state found for isol